# RX Strategist Pipeline Demo

Run RAG evidence retrieval, the drug knowledge graph, the LangGraph workflow, the final checker, and the evaluation set.

This notebook uses a structured prescription, so Gemini is optional. Add `GEMINI_API_KEY` to Colab Secrets only if you want the optional extract cell.

In [ ]:
!git clone https://github.com/YOUR_USERNAME/rx-strategist-mvp.git
%pip install -r rx-strategist-mvp/requirements.txt
import sys
sys.path.insert(0, "rx-strategist-mvp/src")

In [ ]:
from rx_strategist.agents.workflow import run_workflow
from rx_strategist.evaluation.runner import evaluate
from rx_strategist.models.prescription import Medication, Patient, Prescription

prescription = Prescription(
    patient=Patient(
        age=55,
        gender="female",
        conditions=["hypertension", "type 2 diabetes"],
        allergies=[],
        kidney_function="normal",
    ),
    medications=[
        Medication(drug="losartan", dose="50 mg", frequency="once daily", route="oral"),
        Medication(drug="metformin", dose="500 mg", frequency="twice daily", route="oral"),
    ],
)
prescription

In [ ]:
result = run_workflow(prescription=prescription)
{
    "verification": result["verification"]["overall_status"],
    "final_decision": result["final_check"]["final_decision"],
    "evidence": [hit["title"] for hit in result["evidence"]],
    "interactions": result["kg_context"]["interactions"],
}

In [ ]:
result["evidence"]

In [ ]:
result["final_check"]

In [ ]:
report = evaluate()
report

In [ ]:
# Optional: extract from text with Gemini, then run the same workflow.
from google.colab import userdata
from rx_strategist.extraction.gemini_extractor import GeminiPrescriptionExtractor

api_key = userdata.get("GEMINI_API_KEY")
extractor = GeminiPrescriptionExtractor(api_key=api_key)
raw_prescription = """
55-year-old female with hypertension and type 2 diabetes.
Losartan 50 mg OD PO.
Metformin 500 mg BID PO.
Kidney function is normal. No known allergies.
"""
extracted = extractor.extract_prescription(raw_prescription)
run_workflow(prescription=extracted)["final_check"]["final_decision"]